In [10]:
from pathlib import Path
import duckdb

parquet_path = r"../data/processed/20*.parquet"

analysis_start_date = "2019-12-01"
analysis_end_date = "2020-05-01"

anomaly_dates = [
    "2020-01-01",
    "2020-01-02",
    "2020-01-03",
    "2020-02-27",
    "2020-04-20",
    "2020-04-21"
]

anomaly_dates_sql = ", ".join(
    f"DATE '{d}'" for d in anomaly_dates
)

temp_dir = Path(r"D:\duckdb_temp")
temp_dir.mkdir(exist_ok=True)

duckdb.sql("SET temp_directory = 'D:/duckdb_temp'")
duckdb.sql("SET preserve_insertion_order = false")
duckdb.sql("SET threads = 1")
duckdb.sql("SET memory_limit = '6GB'")

In [12]:
# 하나의 product_id가 여러 category_code에 연결되는 경우가 있는지 확인

duckdb.sql(f"""
    WITH product_category AS (
        SELECT
            product_id,
            COUNT(DISTINCT category_code) AS category_count

        FROM read_parquet('{parquet_path}')

        WHERE category_code IS NOT NULL
          AND event_time >= '2019-12-01'
          AND event_time < '2020-05-01'

        GROUP BY product_id
    )

    SELECT
        category_count,
        COUNT(*) AS product_count

    FROM product_category

    GROUP BY category_count
    ORDER BY category_count
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬───────────────┐
│ category_count │ product_count │
│     int64      │     int64     │
├────────────────┼───────────────┤
│              1 │        303323 │
│              2 │          6844 │
└────────────────┴───────────────┘



In [13]:
# 여러 category_code에 연결된 상품의 실제 매핑 사례 확인

duckdb.sql(f"""
    SELECT
        product_id,
        category_code,
        COUNT(*) AS event_count

    FROM read_parquet('{parquet_path}')

    WHERE category_code IS NOT NULL
      AND event_time >= '2019-12-01'
      AND event_time < '2020-05-01'

    GROUP BY
        product_id,
        category_code

    HAVING product_id IN (
        SELECT
            product_id

        FROM read_parquet('{parquet_path}')

        WHERE category_code IS NOT NULL
          AND event_time >= '2019-12-01'
          AND event_time < '2020-05-01'

        GROUP BY product_id
        HAVING COUNT(DISTINCT category_code) > 1
    )

    ORDER BY
        product_id,
        event_count DESC

    LIMIT 40
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────────────────┬─────────────┐
│ product_id │          category_code           │ event_count │
│   int64    │             varchar              │    int64    │
├────────────┼──────────────────────────────────┼─────────────┤
│    1002396 │ construction.tools.light         │          84 │
│    1002396 │ electronics.smartphone           │          48 │
│    1002398 │ electronics.smartphone           │        1028 │
│    1002398 │ construction.tools.light         │         827 │
│    1002415 │ construction.tools.light         │          20 │
│    1002415 │ electronics.smartphone           │          19 │
│    1002522 │ electronics.smartphone           │        1741 │
│    1002522 │ construction.tools.light         │          90 │
│    1002541 │ construction.tools.light         │         403 │
│    1002541 │ electronics.smartphone           │          13 │
│       ·    │           ·                      │           · │
│       ·    │           ·              

In [14]:
# 여러 카테고리에 연결된 상품이 월별로 어떻게 분류되는지 확인

duckdb.sql(f"""
    SELECT
        product_id,
        STRFTIME(event_time, '%Y-%m') AS month,
        category_code,
        COUNT(*) AS event_count

    FROM read_parquet('{parquet_path}')

    WHERE product_id IN (
        1002396,
        1002398,
        1002415,
        1002522,
        1002541,
        1003114,
        1003141,
        1003153
    )
      AND category_code IS NOT NULL
      AND event_time >= '2019-12-01'
      AND event_time < '2020-05-01'

    GROUP BY
        product_id,
        month,
        category_code

    ORDER BY
        product_id,
        month,
        event_count DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────┬──────────────────────────────────┬─────────────┐
│ product_id │  month  │          category_code           │ event_count │
│   int64    │ varchar │             varchar              │    int64    │
├────────────┼─────────┼──────────────────────────────────┼─────────────┤
│    1002396 │ 2019-12 │ electronics.smartphone           │          24 │
│    1002396 │ 2020-01 │ electronics.smartphone           │          24 │
│    1002396 │ 2020-02 │ construction.tools.light         │          52 │
│    1002396 │ 2020-03 │ construction.tools.light         │          18 │
│    1002396 │ 2020-04 │ construction.tools.light         │          14 │
│    1002398 │ 2019-12 │ electronics.smartphone           │         386 │
│    1002398 │ 2020-01 │ electronics.smartphone           │         642 │
│    1002398 │ 2020-02 │ construction.tools.light         │          70 │
│    1002398 │ 2020-03 │ construction.tools.light         │         694 │
│    1002398 │ 2020-04 │ construction.

In [15]:
# 카테고리별 View 이벤트 규모 확인

duckdb.sql(f"""
    SELECT
        category_code,
        COUNT(*) AS view_count

    FROM read_parquet('{parquet_path}')

    WHERE event_type = 'view'
      AND category_code IS NOT NULL
      AND event_time >= '2019-12-01'
      AND event_time < '2020-05-01'

    GROUP BY category_code

    ORDER BY view_count DESC

    LIMIT 30
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────┬────────────┐
│          category_code           │ view_count │
│             varchar              │   int64    │
├──────────────────────────────────┼────────────┤
│ construction.tools.light         │   66097101 │
│ appliances.personal.massager     │    9972318 │
│ sport.bicycle                    │    9789407 │
│ electronics.clocks               │    9528214 │
│ electronics.audio.headphone      │    9514983 │
│ appliances.kitchen.refrigerators │    9310751 │
│ apparel.shoes                    │    8832353 │
│ appliances.environment.vacuum    │    5811765 │
│ computers.peripherals.printer    │    5188742 │
│ appliances.kitchen.washer        │    4025331 │
│       ·                          │       ·    │
│       ·                          │       ·    │
│       ·                          │       ·    │
│ apparel.shorts                   │    2580515 │
│ apparel.scarf                    │    2542430 │
│ furniture.bedroom.bed            │    2405711 │


In [16]:
# 분석 가능한 충분한 규모의 카테고리 수 확인

duckdb.sql(f"""
    WITH category_views AS (
        SELECT
            category_code,
            COUNT(*) AS view_count

        FROM read_parquet('{parquet_path}')

        WHERE event_type = 'view'
          AND category_code IS NOT NULL
          AND event_time >= '2019-12-01'
          AND event_time < '2020-05-01'

        GROUP BY category_code
    )

    SELECT
        COUNT(*) AS category_count,
        MIN(view_count) AS min_view_count,
        MAX(view_count) AS max_view_count

    FROM category_views

    WHERE view_count >= 1000000
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬────────────────┬────────────────┐
│ category_count │ min_view_count │ max_view_count │
│     int64      │     int64      │     int64      │
├────────────────┼────────────────┼────────────────┤
│             56 │        1001366 │       66097101 │
└────────────────┴────────────────┴────────────────┘



In [18]:
# [분석용 코드]
# 30분 이상 활동 공백을 기준으로 분석 세션 재구성
# 이후 카테고리 분석을 위해 category_code도 함께 저장

sessionized_events_parquet = r"../data/processed/sessionized_events.parquet"

Path(sessionized_events_parquet).unlink(missing_ok=True)

duckdb.sql(f"""
COPY (
    WITH ordered_events AS (
        SELECT
            user_id,
            user_session,
            event_time,
            event_type,
            product_id,
            category_code,

            LAG(event_time) OVER (
                PARTITION BY user_id, user_session
                ORDER BY event_time
            ) AS previous_event_time

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND event_time >= '{analysis_start_date}'
          AND event_time < '{analysis_end_date}'
          AND CAST(event_time AS DATE) NOT IN ({anomaly_dates_sql})
    ),

    session_flags AS (
        SELECT
            *,

            CASE
                WHEN previous_event_time IS NULL
                  OR DATE_DIFF(
                        'second',
                        previous_event_time,
                        event_time
                     ) >= 1800
                THEN 1
                ELSE 0
            END AS new_session_flag

        FROM ordered_events
    )

    SELECT
        user_id,
        user_session,
        event_time,
        event_type,
        product_id,
        category_code,

        SUM(new_session_flag) OVER (
            PARTITION BY user_id, user_session
            ORDER BY event_time
            RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS analysis_session_number

    FROM session_flags
)
TO '{sessionized_events_parquet}'
(FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [19]:
# 최초 View 시점의 category_code를 연결하고 중복 행 제거
# 2칸 아래의 코드 결과로 중복된 모든 퍼널에서 category_code 자체는 동일했음을 확인(같은 최초 View 시각에 동일한 View 로그가 여러 행 존재해서 JOIN 결과만 중복된 것)
# 카테고리를 임의로 선택할 필요는 없고, 완전히 동일한 퍼널×카테고리 중복만 제거하면 됨
# 기존 코드를 SELECT 에서 SELECT DISTINCT 로 수정함.
first_view_category_parquet = r"../data/processed/first_view_category.parquet"

Path(first_view_category_parquet).unlink(missing_ok=True)

duckdb.sql(f"""
    COPY (
        SELECT DISTINCT
            f.user_id,
            f.user_session,
            f.analysis_session_number,
            f.product_id,
            f.first_view_time,
            e.category_code

        FROM read_parquet('../data/processed/first_view.parquet') f

        JOIN read_parquet('../data/processed/sessionized_events.parquet') e
            ON f.user_id = e.user_id
           AND f.user_session = e.user_session
           AND f.analysis_session_number = e.analysis_session_number
           AND f.product_id = e.product_id
           AND f.first_view_time = e.event_time

        WHERE e.event_type = 'view'
          AND e.category_code IS NOT NULL
    )
    TO '{first_view_category_parquet}'
    (FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [20]:
# 하나의 퍼널 시작점에 중복된 category_code가 붙었는지 확인
# 위의 코드를 수정했으므로 현재 코드의 결과도 total_rows와 unique_funnels가 같게 나옴.

duckdb.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT (
            user_id,
            user_session,
            analysis_session_number,
            product_id
        )) AS unique_funnels

    FROM read_parquet('{first_view_category_parquet}')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────┐
│ total_rows │ unique_funnels │
│   int64    │     int64      │
├────────────┼────────────────┤
│  166985780 │      166985780 │
└────────────┴────────────────┘



In [21]:
# 중복된 퍼널에 서로 다른 category_code가 붙는 경우가 있는지 확인

duckdb.sql(f"""
    WITH funnel_category_check AS (
        SELECT
            user_id,
            user_session,
            analysis_session_number,
            product_id,

            COUNT(*) AS row_count,
            COUNT(DISTINCT category_code) AS category_count

        FROM read_parquet('{first_view_category_parquet}')

        GROUP BY
            user_id,
            user_session,
            analysis_session_number,
            product_id
    )

    SELECT
        category_count,
        COUNT(*) AS funnel_count

    FROM funnel_category_check

    WHERE row_count > 1

    GROUP BY category_count
    ORDER BY category_count
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬──────────────┐
│ category_count │ funnel_count │
│     int64      │    int64     │
└────────────────┴──────────────┘
             0 rows            



In [23]:
# 카테고리별 View → Cart → Purchase 퍼널 전환율 계산

duckdb.sql(f"""
    WITH funnel_by_category AS (
        SELECT
            v.category_code,

            COUNT(*) AS viewed_count,

            SUM(
                CASE
                    WHEN c.first_cart_after_view IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS carted_count,

            SUM(
                CASE
                    WHEN p.first_purchase_after_cart IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS purchased_count

        FROM read_parquet('{first_view_category_parquet}') v

        LEFT JOIN read_parquet('../data/processed/first_cart_after_view.parquet') c
            ON v.user_id = c.user_id
           AND v.user_session = c.user_session
           AND v.analysis_session_number = c.analysis_session_number
           AND v.product_id = c.product_id

        LEFT JOIN read_parquet('../data/processed/session_product_sequential_funnel.parquet') p
            ON v.user_id = p.user_id
           AND v.user_session = p.user_session
           AND v.analysis_session_number = p.analysis_session_number
           AND v.product_id = p.product_id

        GROUP BY v.category_code
    )

    SELECT
        category_code,
        viewed_count,
        carted_count,
        purchased_count,

        ROUND(
            carted_count * 100.0 / viewed_count,
            2
        ) AS view_to_cart_rate,

        ROUND(
            purchased_count * 100.0 / NULLIF(carted_count, 0),
            2
        ) AS cart_to_purchase_rate,

        ROUND(
            purchased_count * 100.0 / viewed_count,
            2
        ) AS sequential_funnel_completion_rate

    FROM funnel_by_category

    WHERE viewed_count >= 1000000

    ORDER BY viewed_count DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────┬──────────────┬──────────────┬─────────────────┬───────────────────┬───────────────────────┬───────────────────────────────────┐
│          category_code           │ viewed_count │ carted_count │ purchased_count │ view_to_cart_rate │ cart_to_purchase_rate │ sequential_funnel_completion_rate │
│             varchar              │    int64     │    int128    │     int128      │      double       │        double         │              double               │
├──────────────────────────────────┼──────────────┼──────────────┼─────────────────┼───────────────────┼───────────────────────┼───────────────────────────────────┤
│ construction.tools.light         │     42690934 │      3426109 │         1894628 │              8.03 │                  55.3 │                              4.44 │
│ electronics.clocks               │      6424142 │       248641 │          122262 │              3.87 │                 49.17 │                               1.9 │
│ sport.bi

In [24]:
# View→Cart 전환율이 가장 낮은 카테고리 확인

duckdb.sql(f"""
    WITH funnel_by_category AS (
        SELECT
            v.category_code,
            COUNT(*) AS viewed_count,

            SUM(
                CASE
                    WHEN c.first_cart_after_view IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS carted_count,

            SUM(
                CASE
                    WHEN p.first_purchase_after_cart IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS purchased_count

        FROM read_parquet('{first_view_category_parquet}') v

        LEFT JOIN read_parquet('../data/processed/first_cart_after_view.parquet') c
            ON v.user_id = c.user_id
           AND v.user_session = c.user_session
           AND v.analysis_session_number = c.analysis_session_number
           AND v.product_id = c.product_id

        LEFT JOIN read_parquet('../data/processed/session_product_sequential_funnel.parquet') p
            ON v.user_id = p.user_id
           AND v.user_session = p.user_session
           AND v.analysis_session_number = p.analysis_session_number
           AND v.product_id = p.product_id

        GROUP BY v.category_code
    )

    SELECT
        category_code,
        viewed_count,
        ROUND(carted_count * 100.0 / viewed_count, 2) AS view_to_cart_rate

    FROM funnel_by_category

    WHERE viewed_count >= 1000000

    ORDER BY view_to_cart_rate ASC

    LIMIT 10
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────────┬──────────────┬───────────────────┐
│       category_code        │ viewed_count │ view_to_cart_rate │
│          varchar           │    int64     │      double       │
├────────────────────────────┼──────────────┼───────────────────┤
│ apparel.shorts             │      1884142 │              1.92 │
│ construction.tools.drill   │      1669175 │              1.98 │
│ furniture.living_room.sofa │      1169244 │              2.16 │
│ electronics.smartphone     │      1114322 │              2.28 │
│ apparel.underwear          │      1549035 │              2.32 │
│ accessories.bag            │      2182883 │              2.48 │
│ kids.toys                  │      2675481 │               2.7 │
│ apparel.shirt              │      1684442 │               2.7 │
│ sport.trainer              │      2431616 │              2.76 │
│ kids.skates                │      1453488 │              2.77 │
└────────────────────────────┴──────────────┴───────────────────┘
  10 rows 

In [25]:
# View→Cart 전환율이 가장 높은 카테고리 확인

duckdb.sql(f"""
    WITH funnel_by_category AS (
        SELECT
            v.category_code,
            COUNT(*) AS viewed_count,

            SUM(
                CASE
                    WHEN c.first_cart_after_view IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS carted_count,

            SUM(
                CASE
                    WHEN p.first_purchase_after_cart IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS purchased_count

        FROM read_parquet('{first_view_category_parquet}') v

        LEFT JOIN read_parquet('../data/processed/first_cart_after_view.parquet') c
            ON v.user_id = c.user_id
           AND v.user_session = c.user_session
           AND v.analysis_session_number = c.analysis_session_number
           AND v.product_id = c.product_id

        LEFT JOIN read_parquet('../data/processed/session_product_sequential_funnel.parquet') p
            ON v.user_id = p.user_id
           AND v.user_session = p.user_session
           AND v.analysis_session_number = p.analysis_session_number
           AND v.product_id = p.product_id

        GROUP BY v.category_code
    )

    SELECT
        category_code,
        viewed_count,
        ROUND(carted_count * 100.0 / viewed_count, 2) AS view_to_cart_rate

    FROM funnel_by_category

    WHERE viewed_count >= 1000000

    ORDER BY view_to_cart_rate DESC

    LIMIT 10
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────────────────────┬──────────────┬───────────────────┐
│           category_code           │ viewed_count │ view_to_cart_rate │
│              varchar              │    int64     │      double       │
├───────────────────────────────────┼──────────────┼───────────────────┤
│ construction.tools.light          │     42690934 │              8.03 │
│ sport.bicycle                     │      6418797 │              7.13 │
│ appliances.kitchen.washer         │      2365501 │               5.8 │
│ appliances.kitchen.blender        │      1012072 │              5.54 │
│ appliances.personal.massager      │      5952240 │              5.48 │
│ appliances.environment.vacuum     │      3324952 │              5.18 │
│ furniture.bedroom.blanket         │      1709574 │              5.06 │
│ apparel.shoes                     │      5914362 │              4.68 │
│ appliances.kitchen.coffee_grinder │      1394353 │              4.61 │
│ appliances.kitchen.oven           │      1051778 

In [26]:
# Cart→Purchase 전환율이 가장 낮은 카테고리 확인

duckdb.sql(f"""
    WITH funnel_by_category AS (
        SELECT
            v.category_code,
            COUNT(*) AS viewed_count,

            SUM(
                CASE
                    WHEN c.first_cart_after_view IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS carted_count,

            SUM(
                CASE
                    WHEN p.first_purchase_after_cart IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS purchased_count

        FROM read_parquet('{first_view_category_parquet}') v

        LEFT JOIN read_parquet('../data/processed/first_cart_after_view.parquet') c
            ON v.user_id = c.user_id
           AND v.user_session = c.user_session
           AND v.analysis_session_number = c.analysis_session_number
           AND v.product_id = c.product_id

        LEFT JOIN read_parquet('../data/processed/session_product_sequential_funnel.parquet') p
            ON v.user_id = p.user_id
           AND v.user_session = p.user_session
           AND v.analysis_session_number = p.analysis_session_number
           AND v.product_id = p.product_id

        GROUP BY v.category_code
    )

    SELECT
        category_code,
        carted_count,
        ROUND(
            purchased_count * 100.0 / NULLIF(carted_count, 0),
            2
        ) AS cart_to_purchase_rate

    FROM funnel_by_category

    WHERE viewed_count >= 1000000

    ORDER BY cart_to_purchase_rate ASC

    LIMIT 10
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────────┬──────────────┬───────────────────────┐
│       category_code        │ carted_count │ cart_to_purchase_rate │
│          varchar           │    int128    │        double         │
├────────────────────────────┼──────────────┼───────────────────────┤
│ apparel.scarf              │        65764 │                 29.15 │
│ appliances.kitchen.grill   │        33959 │                 34.15 │
│ apparel.underwear          │        35887 │                 34.73 │
│ furniture.living_room.sofa │        25254 │                 34.97 │
│ sport.trainer              │        67131 │                 35.29 │
│ accessories.bag            │        54191 │                 36.26 │
│ computers.desktop          │        43232 │                 36.73 │
│ furniture.kitchen.table    │        81741 │                 37.28 │
│ apparel.shirt              │        45480 │                 38.69 │
│ apparel.costume            │        48205 │                 39.36 │
└───────────────────

In [27]:
# Cart→Purchase 전환율이 가장 낮은 카테고리 확인

duckdb.sql(f"""
    WITH funnel_by_category AS (
        SELECT
            v.category_code,
            COUNT(*) AS viewed_count,

            SUM(
                CASE
                    WHEN c.first_cart_after_view IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS carted_count,

            SUM(
                CASE
                    WHEN p.first_purchase_after_cart IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS purchased_count

        FROM read_parquet('{first_view_category_parquet}') v

        LEFT JOIN read_parquet('../data/processed/first_cart_after_view.parquet') c
            ON v.user_id = c.user_id
           AND v.user_session = c.user_session
           AND v.analysis_session_number = c.analysis_session_number
           AND v.product_id = c.product_id

        LEFT JOIN read_parquet('../data/processed/session_product_sequential_funnel.parquet') p
            ON v.user_id = p.user_id
           AND v.user_session = p.user_session
           AND v.analysis_session_number = p.analysis_session_number
           AND v.product_id = p.product_id

        GROUP BY v.category_code
    )

    SELECT
        category_code,
        carted_count,
        ROUND(
            purchased_count * 100.0 / NULLIF(carted_count, 0),
            2
        ) AS cart_to_purchase_rate

    FROM funnel_by_category

    WHERE viewed_count >= 1000000

    ORDER BY cart_to_purchase_rate DESC

    LIMIT 10
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────────────────────┬──────────────┬───────────────────────┐
│           category_code           │ carted_count │ cart_to_purchase_rate │
│              varchar              │    int128    │        double         │
├───────────────────────────────────┼──────────────┼───────────────────────┤
│ appliances.kitchen.coffee_grinder │        64318 │                 59.73 │
│ construction.tools.light          │      3426109 │                  55.3 │
│ construction.components.faucet    │        71968 │                 52.62 │
│ computers.peripherals.printer     │       112731 │                 50.44 │
│ electronics.audio.headphone       │       231714 │                 49.56 │
│ appliances.environment.vacuum     │       172083 │                 49.18 │
│ electronics.clocks                │       248641 │                 49.17 │
│ appliances.personal.massager      │       326089 │                 48.51 │
│ appliances.kitchen.washer         │       137107 │                  48.5 │

In [28]:
# 3월 → 4월 카테고리별 Cart→Purchase 전환율 변화 확인

duckdb.sql(f"""
    WITH category_monthly AS (
        SELECT
            STRFTIME(v.first_view_time, '%Y-%m') AS month,
            v.category_code,

            COUNT(*) AS viewed_count,

            SUM(
                CASE
                    WHEN c.first_cart_after_view IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS carted_count,

            SUM(
                CASE
                    WHEN p.first_purchase_after_cart IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS purchased_count

        FROM read_parquet('{first_view_category_parquet}') v

        LEFT JOIN read_parquet('../data/processed/first_cart_after_view.parquet') c
            ON v.user_id = c.user_id
           AND v.user_session = c.user_session
           AND v.analysis_session_number = c.analysis_session_number
           AND v.product_id = c.product_id

        LEFT JOIN read_parquet('../data/processed/session_product_sequential_funnel.parquet') p
            ON v.user_id = p.user_id
           AND v.user_session = p.user_session
           AND v.analysis_session_number = p.analysis_session_number
           AND v.product_id = p.product_id

        WHERE STRFTIME(v.first_view_time, '%Y-%m') IN ('2020-03', '2020-04')

        GROUP BY
            month,
            v.category_code
    ),

    category_rate AS (
        SELECT
            month,
            category_code,
            viewed_count,
            carted_count,
            purchased_count,

            purchased_count * 100.0
            / NULLIF(carted_count, 0) AS cart_to_purchase_rate

        FROM category_monthly
    ),

    comparison AS (
        SELECT
            category_code,

            MAX(CASE
                WHEN month = '2020-03'
                THEN carted_count
            END) AS march_carted_count,

            MAX(CASE
                WHEN month = '2020-04'
                THEN carted_count
            END) AS april_carted_count,

            MAX(CASE
                WHEN month = '2020-03'
                THEN cart_to_purchase_rate
            END) AS march_rate,

            MAX(CASE
                WHEN month = '2020-04'
                THEN cart_to_purchase_rate
            END) AS april_rate

        FROM category_rate

        GROUP BY category_code
    )

    SELECT
        category_code,
        march_carted_count,
        april_carted_count,

        ROUND(march_rate, 2) AS march_rate,
        ROUND(april_rate, 2) AS april_rate,

        ROUND(
            april_rate - march_rate,
            2
        ) AS change_pp

    FROM comparison

    WHERE march_carted_count >= 10000
      AND april_carted_count >= 10000

    ORDER BY change_pp ASC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────┬────────────────────┬────────────────────┬────────────┬────────────┬───────────┐
│          category_code           │ march_carted_count │ april_carted_count │ march_rate │ april_rate │ change_pp │
│             varchar              │       int128       │       int128       │   double   │   double   │  double   │
├──────────────────────────────────┼────────────────────┼────────────────────┼────────────┼────────────┼───────────┤
│ computers.peripherals.printer    │              27702 │              13510 │      55.82 │      46.12 │      -9.7 │
│ construction.tools.light         │             664556 │             639602 │      59.58 │      50.05 │     -9.53 │
│ electronics.clocks               │              47544 │              37739 │      53.21 │      44.52 │     -8.69 │
│ apparel.trousers                 │              10229 │              10945 │      51.32 │      43.77 │     -7.55 │
│ sport.bicycle                    │              93869 │       

In [29]:
# 3월 전환율이 4월에도 유지됐다고 가정했을 때의 예상 Purchase와 실제 Purchase 차이 계산

duckdb.sql(f"""
    WITH category_monthly AS (
        SELECT
            STRFTIME(v.first_view_time, '%Y-%m') AS month,
            v.category_code,

            SUM(
                CASE
                    WHEN c.first_cart_after_view IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS carted_count,

            SUM(
                CASE
                    WHEN p.first_purchase_after_cart IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS purchased_count

        FROM read_parquet('{first_view_category_parquet}') v

        LEFT JOIN read_parquet('../data/processed/first_cart_after_view.parquet') c
            ON v.user_id = c.user_id
           AND v.user_session = c.user_session
           AND v.analysis_session_number = c.analysis_session_number
           AND v.product_id = c.product_id

        LEFT JOIN read_parquet('../data/processed/session_product_sequential_funnel.parquet') p
            ON v.user_id = p.user_id
           AND v.user_session = p.user_session
           AND v.analysis_session_number = p.analysis_session_number
           AND v.product_id = p.product_id

        WHERE STRFTIME(v.first_view_time, '%Y-%m') IN ('2020-03', '2020-04')

        GROUP BY
            month,
            v.category_code
    ),

    comparison AS (
        SELECT
            category_code,

            MAX(CASE
                WHEN month = '2020-03'
                THEN carted_count
            END) AS march_carted_count,

            MAX(CASE
                WHEN month = '2020-03'
                THEN purchased_count
            END) AS march_purchased_count,

            MAX(CASE
                WHEN month = '2020-04'
                THEN carted_count
            END) AS april_carted_count,

            MAX(CASE
                WHEN month = '2020-04'
                THEN purchased_count
            END) AS april_purchased_count

        FROM category_monthly

        GROUP BY category_code
    ),

    impact AS (
        SELECT
            category_code,
            march_carted_count,
            march_purchased_count,
            april_carted_count,
            april_purchased_count,

            march_purchased_count * 1.0
            / NULLIF(march_carted_count, 0) AS march_rate,

            april_purchased_count * 1.0
            / NULLIF(april_carted_count, 0) AS april_rate

        FROM comparison

        WHERE march_carted_count >= 10000
          AND april_carted_count >= 10000
    )

    SELECT
        category_code,

        april_carted_count,

        ROUND(march_rate * 100, 2) AS march_rate,
        ROUND(april_rate * 100, 2) AS april_rate,

        ROUND(
            april_carted_count * march_rate,
            0
        ) AS expected_april_purchases,

        april_purchased_count AS actual_april_purchases,

        ROUND(
            april_carted_count * march_rate
            - april_purchased_count,
            0
        ) AS purchase_shortfall

    FROM impact

    ORDER BY purchase_shortfall DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────┬────────────────────┬────────────┬────────────┬──────────────────────────┬────────────────────────┬────────────────────┐
│          category_code           │ april_carted_count │ march_rate │ april_rate │ expected_april_purchases │ actual_april_purchases │ purchase_shortfall │
│             varchar              │       int128       │   double   │   double   │          double          │         int128         │       double       │
├──────────────────────────────────┼────────────────────┼────────────┼────────────┼──────────────────────────┼────────────────────────┼────────────────────┤
│ construction.tools.light         │             639602 │      59.58 │      50.05 │                 381074.0 │                 320115 │            60959.0 │
│ sport.bicycle                    │              77152 │      50.77 │      44.64 │                  39170.0 │                  34444 │             4726.0 │
│ electronics.audio.headphone      │              72301 │ 

In [30]:
# [분석용 코드]
# 4월 Purchase 부족분의 카테고리별 기여 비중 계산

duckdb.sql(f"""
    WITH category_monthly AS (
        SELECT
            STRFTIME(v.first_view_time, '%Y-%m') AS month,
            v.category_code,

            SUM(
                CASE
                    WHEN c.first_cart_after_view IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS carted_count,

            SUM(
                CASE
                    WHEN p.first_purchase_after_cart IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS purchased_count

        FROM read_parquet('{first_view_category_parquet}') v

        LEFT JOIN read_parquet('../data/processed/first_cart_after_view.parquet') c
            ON v.user_id = c.user_id
           AND v.user_session = c.user_session
           AND v.analysis_session_number = c.analysis_session_number
           AND v.product_id = c.product_id

        LEFT JOIN read_parquet('../data/processed/session_product_sequential_funnel.parquet') p
            ON v.user_id = p.user_id
           AND v.user_session = p.user_session
           AND v.analysis_session_number = p.analysis_session_number
           AND v.product_id = p.product_id

        WHERE STRFTIME(v.first_view_time, '%Y-%m') IN ('2020-03', '2020-04')

        GROUP BY
            month,
            v.category_code
    ),

    comparison AS (
        SELECT
            category_code,

            MAX(CASE
                WHEN month = '2020-03'
                THEN carted_count
            END) AS march_carted_count,

            MAX(CASE
                WHEN month = '2020-03'
                THEN purchased_count
            END) AS march_purchased_count,

            MAX(CASE
                WHEN month = '2020-04'
                THEN carted_count
            END) AS april_carted_count,

            MAX(CASE
                WHEN month = '2020-04'
                THEN purchased_count
            END) AS april_purchased_count

        FROM category_monthly

        GROUP BY category_code
    ),

    shortfall AS (
        SELECT
            category_code,

            april_carted_count,

            march_purchased_count * 1.0
            / NULLIF(march_carted_count, 0) AS march_rate,

            april_purchased_count,

            april_carted_count
            * (
                march_purchased_count * 1.0
                / NULLIF(march_carted_count, 0)
            )
            - april_purchased_count AS purchase_shortfall

        FROM comparison

        WHERE march_carted_count >= 10000
          AND april_carted_count >= 10000
    ),

    positive_shortfall AS (
        SELECT *
        FROM shortfall
        WHERE purchase_shortfall > 0
    )

    SELECT
        category_code,

        ROUND(purchase_shortfall, 0) AS purchase_shortfall,

        ROUND(
            purchase_shortfall * 100.0
            / SUM(purchase_shortfall) OVER (),
            2
        ) AS shortfall_share_pct

    FROM positive_shortfall

    ORDER BY purchase_shortfall DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────┬────────────────────┬─────────────────────┐
│          category_code           │ purchase_shortfall │ shortfall_share_pct │
│             varchar              │       double       │       double        │
├──────────────────────────────────┼────────────────────┼─────────────────────┤
│ construction.tools.light         │            60959.0 │               70.76 │
│ sport.bicycle                    │             4726.0 │                5.49 │
│ electronics.audio.headphone      │             3703.0 │                 4.3 │
│ appliances.personal.massager     │             3467.0 │                4.02 │
│ electronics.clocks               │             3278.0 │                3.81 │
│ apparel.shoes                    │             1365.0 │                1.58 │
│ computers.peripherals.printer    │             1310.0 │                1.52 │
│ apparel.shoes.slipons            │             1129.0 │                1.31 │
│ appliances.environment.vacuum    │    

In [31]:
# construction.tools.light의 월별 이벤트 및 상품 규모 변화 확인

duckdb.sql(f"""
    SELECT
        STRFTIME(event_time, '%Y-%m') AS month,

        COUNT(*) AS total_events,

        SUM(
            CASE WHEN event_type = 'view'
                 THEN 1 ELSE 0 END
        ) AS view_count,

        SUM(
            CASE WHEN event_type = 'cart'
                 THEN 1 ELSE 0 END
        ) AS cart_count,

        SUM(
            CASE WHEN event_type = 'purchase'
                 THEN 1 ELSE 0 END
        ) AS purchase_count,

        COUNT(DISTINCT product_id) AS product_count

    FROM read_parquet('{parquet_path}')

    WHERE category_code = 'construction.tools.light'
      AND event_time >= '2019-12-01'
      AND event_time < '2020-05-01'

    GROUP BY month
    ORDER BY month
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬──────────────┬────────────┬────────────┬────────────────┬───────────────┐
│  month  │ total_events │ view_count │ cart_count │ purchase_count │ product_count │
│ varchar │    int64     │   int128   │   int128   │     int128     │     int64     │
├─────────┼──────────────┼────────────┼────────────┼────────────────┼───────────────┤
│ 2019-12 │     16519261 │   14696444 │    1320744 │         502073 │          2021 │
│ 2020-01 │     15090920 │   13560829 │    1124033 │         406058 │          2190 │
│ 2020-02 │     14866151 │   13091404 │    1184886 │         589861 │          2494 │
│ 2020-03 │     13526546 │   11966456 │    1115841 │         444249 │          2468 │
│ 2020-04 │     14325436 │   12781968 │    1172824 │         370644 │          2418 │
└─────────┴──────────────┴────────────┴────────────┴────────────────┴───────────────┘



In [32]:
# construction.tools.light의 3~4월 일별 Cart→Purchase 전환율 확인

duckdb.sql(f"""
    WITH daily_category AS (
        SELECT
            CAST(v.first_view_time AS DATE) AS event_date,

            SUM(
                CASE
                    WHEN c.first_cart_after_view IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS carted_count,

            SUM(
                CASE
                    WHEN p.first_purchase_after_cart IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS purchased_count

        FROM read_parquet('{first_view_category_parquet}') v

        LEFT JOIN read_parquet('../data/processed/first_cart_after_view.parquet') c
            ON v.user_id = c.user_id
           AND v.user_session = c.user_session
           AND v.analysis_session_number = c.analysis_session_number
           AND v.product_id = c.product_id

        LEFT JOIN read_parquet('../data/processed/session_product_sequential_funnel.parquet') p
            ON v.user_id = p.user_id
           AND v.user_session = p.user_session
           AND v.analysis_session_number = p.analysis_session_number
           AND v.product_id = p.product_id

        WHERE v.category_code = 'construction.tools.light'
          AND v.first_view_time >= '2020-03-01'
          AND v.first_view_time < '2020-05-01'

        GROUP BY event_date
    )

    SELECT
        event_date,
        carted_count,
        purchased_count,

        ROUND(
            purchased_count * 100.0
            / NULLIF(carted_count, 0),
            2
        ) AS cart_to_purchase_rate

    FROM daily_category

    ORDER BY event_date
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────┬─────────────────┬───────────────────────┐
│ event_date │ carted_count │ purchased_count │ cart_to_purchase_rate │
│    date    │    int128    │     int128      │        double         │
├────────────┼──────────────┼─────────────────┼───────────────────────┤
│ 2020-03-01 │        19303 │           10358 │                 53.66 │
│ 2020-03-02 │        35124 │           20837 │                 59.32 │
│ 2020-03-03 │        31863 │           18640 │                  58.5 │
│ 2020-03-04 │        30569 │           18004 │                  58.9 │
│ 2020-03-05 │        30445 │           17737 │                 58.26 │
│ 2020-03-06 │        30329 │           17479 │                 57.63 │
│ 2020-03-07 │        32597 │           18418 │                  56.5 │
│ 2020-03-08 │        29731 │           17118 │                 57.58 │
│ 2020-03-09 │        28689 │           16672 │                 58.11 │
│ 2020-03-10 │        28425 │           16737 │                 

In [34]:
#------------------------------------------------------------------------------------------------------------------------#

In [35]:
# [Tableau용 저장 파일]
# 주요 카테고리별 View→Cart / Cart→Purchase 퍼널 성과 저장

category_funnel_csv = r"../data/marts/dashboard_category_funnel.csv"

duckdb.sql(f"""
COPY (
    WITH funnel_by_category AS (
        SELECT
            v.category_code,
            COUNT(*) AS viewed_count,

            SUM(
                CASE
                    WHEN c.first_cart_after_view IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS carted_count,

            SUM(
                CASE
                    WHEN p.first_purchase_after_cart IS NOT NULL THEN 1
                    ELSE 0
                END
            ) AS purchased_count

        FROM read_parquet('{first_view_category_parquet}') v

        LEFT JOIN read_parquet(
            '../data/processed/first_cart_after_view.parquet'
        ) c
            ON v.user_id = c.user_id
           AND v.user_session = c.user_session
           AND v.analysis_session_number = c.analysis_session_number
           AND v.product_id = c.product_id

        LEFT JOIN read_parquet(
            '../data/processed/session_product_sequential_funnel.parquet'
        ) p
            ON v.user_id = p.user_id
           AND v.user_session = p.user_session
           AND v.analysis_session_number = p.analysis_session_number
           AND v.product_id = p.product_id

        GROUP BY v.category_code
    )

    SELECT
        category_code,
        viewed_count,
        carted_count,
        purchased_count,

        ROUND(
            carted_count * 100.0 / viewed_count,
            2
        ) AS view_to_cart_rate,

        ROUND(
            purchased_count * 100.0 / NULLIF(carted_count, 0),
            2
        ) AS cart_to_purchase_rate,

        ROUND(
            purchased_count * 100.0 / viewed_count,
            2
        ) AS funnel_completion_rate

    FROM funnel_by_category

    WHERE viewed_count >= 1000000

    ORDER BY viewed_count DESC
)
TO '{category_funnel_csv}'
(HEADER, DELIMITER ',')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [36]:
# 카테고리별 Tableau CSV 확인

duckdb.sql(f"""
    SELECT *
    FROM read_csv_auto('{category_funnel_csv}')
    ORDER BY viewed_count DESC
""").show()

┌──────────────────────────────────┬──────────────┬──────────────┬─────────────────┬───────────────────┬───────────────────────┬────────────────────────┐
│          category_code           │ viewed_count │ carted_count │ purchased_count │ view_to_cart_rate │ cart_to_purchase_rate │ funnel_completion_rate │
│             varchar              │    int64     │    int64     │      int64      │      double       │        double         │         double         │
├──────────────────────────────────┼──────────────┼──────────────┼─────────────────┼───────────────────┼───────────────────────┼────────────────────────┤
│ construction.tools.light         │     42690934 │      3426109 │         1894628 │              8.03 │                  55.3 │                   4.44 │
│ electronics.clocks               │      6424142 │       248641 │          122262 │              3.87 │                 49.17 │                    1.9 │
│ sport.bicycle                    │      6418797 │       457637 │          

In [62]:
# [Tableau용 저장 파일]
# 3월→4월 카테고리별 Cart→Purchase 전환율 변화 저장

category_change_csv = r"../data/marts/dashboard_category_conversion_change.csv"

duckdb.sql(f"""
COPY (
    WITH category_monthly AS (
        SELECT
            STRFTIME(v.first_view_time, '%Y-%m') AS month,
            v.category_code,

            SUM(
                CASE
                    WHEN c.first_cart_after_view IS NOT NULL
                    THEN 1 ELSE 0
                END
            ) AS carted_count,

            SUM(
                CASE
                    WHEN p.first_purchase_after_cart IS NOT NULL
                    THEN 1 ELSE 0
                END
            ) AS purchased_count

        FROM read_parquet('{first_view_category_parquet}') v

        LEFT JOIN read_parquet(
            '../data/processed/first_cart_after_view.parquet'
        ) c
            ON v.user_id = c.user_id
           AND v.user_session = c.user_session
           AND v.analysis_session_number = c.analysis_session_number
           AND v.product_id = c.product_id

        LEFT JOIN read_parquet(
            '../data/processed/session_product_sequential_funnel.parquet'
        ) p
            ON v.user_id = p.user_id
           AND v.user_session = p.user_session
           AND v.analysis_session_number = p.analysis_session_number
           AND v.product_id = p.product_id

        WHERE STRFTIME(v.first_view_time, '%Y-%m')
              IN ('2020-03', '2020-04')

        GROUP BY
            month,
            v.category_code
    ),

    comparison AS (
        SELECT
            category_code,

            MAX(CASE WHEN month = '2020-03'
                THEN carted_count END) AS march_carted_count,

            MAX(CASE WHEN month = '2020-04'
                THEN carted_count END) AS april_carted_count,

            MAX(CASE WHEN month = '2020-03'
                THEN purchased_count * 100.0
                     / NULLIF(carted_count, 0)
            END) AS march_rate,

            MAX(CASE WHEN month = '2020-04'
                THEN purchased_count * 100.0
                     / NULLIF(carted_count, 0)
            END) AS april_rate

        FROM category_monthly

        GROUP BY category_code
    )

    SELECT
        category_code,
        march_carted_count,
        april_carted_count,

        ROUND(march_rate, 2) AS march_rate,
        ROUND(april_rate, 2) AS april_rate,

        ROUND(
            april_rate - march_rate,
            2
        ) AS change_pp

    FROM comparison

    WHERE march_carted_count >= 10000
      AND april_carted_count >= 10000

    ORDER BY change_pp
)
TO '{category_change_csv}'
(HEADER, DELIMITER ',')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [63]:
# [검증용 코드]
# 카테고리별 3월→4월 전환율 변화 CSV 확인

duckdb.sql(f"""
    SELECT *
    FROM read_csv_auto('{category_change_csv}')
    ORDER BY change_pp
""").show()

┌──────────────────────────────────┬────────────────────┬────────────────────┬────────────┬────────────┬───────────┐
│          category_code           │ march_carted_count │ april_carted_count │ march_rate │ april_rate │ change_pp │
│             varchar              │       int64        │       int64        │   double   │   double   │  double   │
├──────────────────────────────────┼────────────────────┼────────────────────┼────────────┼────────────┼───────────┤
│ computers.peripherals.printer    │              27702 │              13510 │      55.82 │      46.12 │      -9.7 │
│ construction.tools.light         │             664556 │             639602 │      59.58 │      50.05 │     -9.53 │
│ electronics.clocks               │              47544 │              37739 │      53.21 │      44.52 │     -8.69 │
│ apparel.trousers                 │              10229 │              10945 │      51.32 │      43.77 │     -7.55 │
│ sport.bicycle                    │              93869 │       